# AT1 — Perceptron, KNN e Recomendador por Similaridade

**Aluno:** Érlon Viana dos Santos
**Disciplina:** Aprendizagem de Máquina
**Data:** 26/09/2026

Este notebook implementa, com NumPy puro, três algoritmos clássicos:

1. **Perceptron treinável** (classificação binária de fraudes)
2. **K-Nearest Neighbors** (predição de churn)
3. **Recomendador por similaridade espacial** (sizing de servidores cloud)

Todas as implementações priorizam operações vetorizadas e são reprodutíveis.

In [1]:
# ============================================================
# 0.0 IMPORTS E CONFIGURAÇÃO GLOBAL
# ============================================================
import numpy as np

# Semente para reprodutibilidade (embora os 3 algoritmos aqui sejam determinísticos)
np.random.seed(42)

print("✅ Ambiente configurado. NumPy versão:", np.__version__)

✅ Ambiente configurado. NumPy versão: 2.5.3


---
# Desafio 1 — Classificação Binária com Perceptron Treinável

**Contexto:** triagem preliminar de transações financeiras com risco de fraude.

**Entradas:**
- $x_1$: valor normalizado da transação (0.0 a 10.0)
- $x_2$: frequência de operações recentes (últimos 5 min)

**Saída:** 0 = Legítima | 1 = Suspeita

**Estratégia:** regra iterativa de Rosenblatt, inicializando pesos e viés com 1.0.

In [2]:
# ============================================================
# 1.0 DESAFIO 1 — DADOS DE TREINO
# ============================================================
#valor e frequencia
X_treino_fraude = np.array([
    [1.5, 1.0],
    [2.0, 2.0],
    [3.5, 1.5],
    [3.0, 3.0],
    [6.5, 5.0],
    [7.0, 7.0],
    [8.5, 6.0],
    [9.0, 8.0]
])

#0 legítimo, 1 fraude
y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])

print(f"X_treino_fraude: shape={X_treino_fraude.shape}, dtype={X_treino_fraude.dtype}")
print(f"y_treino_fraude: shape={y_treino_fraude.shape}, classes={np.unique(y_treino_fraude)}")

X_treino_fraude: shape=(8, 2), dtype=float64
y_treino_fraude: shape=(8,), classes=[0 1]


In [ ]:
# ============================================================
# 1.1 Função de treino
# ============================================================
def treinar_perceptron(X, y, taxa_aprendizado=0.01, epocas=100, verbose=False):
    """
    Treina um Perceptron segundo a regra iterativa de Rosenblatt.

    verbose : bool
        Se True, imprime o estado dos parâmetros ao final de cada época.
    """
    n_amostras, n_features = X.shape

    pesos = np.ones(n_features, dtype=float)
    vies = 1.0

    for epoca in range(epocas):
        erros = 0  # contador de erros por época

        # --- loop interno: uma amostra por vez (regra do enunciado) ---
        for i in range(n_amostras):
            #calulo da ativação: somatorio ponderado das entradas + viés 
            z = np.dot(X[i], pesos) + vies
            #rosenblat: A ideia é simples: quando o modelo erra,
            # ajusta os pesos na direção do erro. Quando acerta, não faz nada.
            # Isso garante convergência para dados linearmente separáveis

            #Se o resultado for maior ou igual a zero, a saída é 1; caso contrário, 0.
            y_pred = 1 if z >= 0 else 0
            #Se a previsão foi correta, erro = 0. Se foi incorreta, erro = ±1
            erro = y[i] - y_pred
            if erro != 0:
                erros += 1
            #atualização dos pesos e viés: se acerta, nada muda; Se errou, os pesos e viés são ajustados na direção da entrada que causou o erro.
            pesos += taxa_aprendizado * erro * X[i]
            vies += taxa_aprendizado * erro

        # imprime estado ao final da época, se verbose=True
        if verbose:
            print(f"Época {epoca+1}/{epocas} | Erros: {erros} | Pesos: {pesos} | Viés: {vies}")

    return pesos, vies

In [4]:
# ============================================================
# 1.2 DESAFIO 1 — FUNÇÃO DE INFERÊNCIA
# ============================================================
def prever_perceptron(x, pesos, vies):
    """Retorna a classe (0 ou 1) para uma única amostra."""
    z = np.dot(x, pesos) + vies
    return 1 if z >= 0 else 0

In [5]:
# ============================================================
# 1.3 DESAFIO 1 — EXECUÇÃO E DIAGNÓSTICO
# ============================================================
pesos_f, vies_f = treinar_perceptron(
    X_treino_fraude, y_treino_fraude,
    taxa_aprendizado=0.01, epocas=100, verbose=True
)

print("=" * 50)
print("PARÂMETROS CALIBRADOS")
print("=" * 50)
print(f"Pesos finais : {pesos_f}")
print(f"Viés final   : {vies_f:.4f}")

# Transações em tempo real
transacao_A = np.array([5.0, 4.0])
transacao_B = np.array([7.0, 6.5])

print("\n" + "=" * 50)
print("DIAGNÓSTICO DAS TRANSAÇÕES")
print("=" * 50)

for nome, t in [("Transação A", transacao_A), ("Transação B", transacao_B)]:
    classe = prever_perceptron(t, pesos_f, vies_f)
    rotulo = "Suspeita (Fraude)" if classe == 1 else "Legítima"
    print(f"{nome} {t.tolist()} -> classe {classe} ({rotulo})")

Época 1/100 | Erros: 4 | Pesos: [0.9   0.925] | Viés: 0.96
Época 2/100 | Erros: 4 | Pesos: [0.8  0.85] | Viés: 0.9199999999999999
Época 3/100 | Erros: 4 | Pesos: [0.7   0.775] | Viés: 0.8799999999999999
Época 4/100 | Erros: 4 | Pesos: [0.6 0.7] | Viés: 0.8399999999999999
Época 5/100 | Erros: 4 | Pesos: [0.5   0.625] | Viés: 0.7999999999999998
Época 6/100 | Erros: 4 | Pesos: [0.4  0.55] | Viés: 0.7599999999999998
Época 7/100 | Erros: 4 | Pesos: [0.3   0.475] | Viés: 0.7199999999999998
Época 8/100 | Erros: 4 | Pesos: [0.2 0.4] | Viés: 0.6799999999999997
Época 9/100 | Erros: 4 | Pesos: [0.1   0.325] | Viés: 0.6399999999999997
Época 10/100 | Erros: 4 | Pesos: [-6.66133815e-16  2.50000000e-01] | Viés: 0.5999999999999996
Época 11/100 | Erros: 4 | Pesos: [-0.1    0.175] | Viés: 0.5599999999999996
Época 12/100 | Erros: 5 | Pesos: [-0.135  0.15 ] | Viés: 0.5299999999999996
Época 13/100 | Erros: 6 | Pesos: [-0.085  0.185] | Viés: 0.5099999999999996
Época 14/100 | Erros: 5 | Pesos: [-0.12  0.16] 

---
# Desafio 2 — Predição de Churn com KNN

**Contexto:** antecipar risco de cancelamento de clientes SaaS.

**Entradas:**
- $x_1$: dias de inatividade (últimos 30 dias)
- $x_2$: chamados críticos em aberto no mês

**Saída:** 0 = Baixo Risco | 1 = Alto Risco

**Estratégia:** distâncias vetorizadas (Euclidiana ou Manhattan) + votação por maioria.

In [6]:
# ============================================================
# 2.0 DESAFIO 2 — DADOS DE TREINO
# ============================================================
X_treino_churn = np.array([
    [2.0, 1.0],
    [3.0, 0.0],
    [5.0, 1.0],
    [6.0, 2.0],
    [15.0, 4.0],
    [18.0, 5.0],
    [20.0, 4.0],
    [22.0, 6.0]
])

y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])

print(f"X_treino_churn: shape={X_treino_churn.shape}")
print(f"y_treino_churn: classes={np.unique(y_treino_churn)}")

X_treino_churn: shape=(8, 2)
y_treino_churn: classes=[0 1]


In [7]:
# ============================================================
# 2.1 DESAFIO 2 — FUNÇÃO DE CLASSIFICAÇÃO KNN
# ============================================================
def knn_classificar(X, y, ponto, k=3, metrica="euclidiana"):
    """
    Classifica um ponto via KNN com cálculo vetorizado de distâncias.

    Parâmetros
    ----------
    X : np.ndarray, shape (n_amostras, n_features)
    y : np.ndarray, shape (n_amostras,)
    ponto : np.ndarray, shape (n_features,)
    k : int
    metrica : {"euclidiana", "manhattan"}

    Retorna
    -------
    classe_predita : int
    indices_vizinhos : np.ndarray
    distancias_vizinhos : np.ndarray
    """
    # --- Cálculo vetorizado das distâncias (sem laço manual) ---
    if metrica == "euclidiana":
        distancias = np.sqrt(np.sum((X - ponto) ** 2, axis=1))
    elif metrica == "manhattan":
        distancias = np.sum(np.abs(X - ponto), axis=1)
    else:
        raise ValueError("Métrica deve ser 'euclidiana' ou 'manhattan'")

    # --- Índices dos k menores ---
    indices_vizinhos = np.argsort(distancias)[:k]
    classes_vizinhos = y[indices_vizinhos]

    # --- Votação por maioria ---
    contagem = np.bincount(classes_vizinhos, minlength=2)
    classe_predita = int(np.argmax(contagem))

    return classe_predita, indices_vizinhos, distancias[indices_vizinhos]

In [8]:
# ============================================================
# 2.2 DESAFIO 2 — EXECUÇÃO E DIAGNÓSTICO
# ============================================================
cliente1 = np.array([13.0, 10.0])
cliente2 = np.array([7.0, 1.5])

print("=" * 60)
print("CLASSIFICAÇÃO DE CHURN — KNN (k=3, Euclidiana)")
print("=" * 60)

for nome, cliente in [("Cliente 1", cliente1), ("Cliente 2", cliente2)]:
    classe, idx, dists = knn_classificar(
        X_treino_churn, y_treino_churn,
        cliente, k=3, metrica="euclidiana"
    )
    rotulo = "Alto Risco" if classe == 1 else "Baixo Risco"

    print(f"\n{nome} {cliente.tolist()}")
    print(f"  Classe predita     : {classe} ({rotulo})")
    print(f"  Índices vizinhos   : {idx.tolist()}")
    print(f"  Amostras vizinhas  : {X_treino_churn[idx].tolist()}")
    print(f"  Classes vizinhas   : {y_treino_churn[idx].tolist()}")
    print(f"  Distâncias         : {np.round(dists, 4).tolist()}")

CLASSIFICAÇÃO DE CHURN — KNN (k=3, Euclidiana)

Cliente 1 [13.0, 10.0]
  Classe predita     : 1 (Alto Risco)
  Índices vizinhos   : [4, 5, 6]
  Amostras vizinhas  : [[15.0, 4.0], [18.0, 5.0], [20.0, 4.0]]
  Classes vizinhas   : [1, 1, 1]
  Distâncias         : [6.3246, 7.0711, 9.2195]

Cliente 2 [7.0, 1.5]
  Classe predita     : 0 (Baixo Risco)
  Índices vizinhos   : [3, 2, 1]
  Amostras vizinhas  : [[6.0, 2.0], [5.0, 1.0], [3.0, 0.0]]
  Classes vizinhas   : [0, 0, 0]
  Distâncias         : [1.118, 2.0616, 4.272]


---
# Desafio 3 — Recomendação de Servidores Cloud

**Contexto:** sizing automático de VMs a partir dos requisitos de uma aplicação.

**Sem rótulos (`y`)** — problema de similaridade espacial.

**Entradas:** [vCPUs, RAM (GB), SSD (GB)]
**Saída:** top-k servidores mais próximos (distância euclidiana).

In [9]:
# ============================================================
# 3.0 DESAFIO 3 — CATÁLOGO DE SERVIDORES
# ============================================================
catalogo_servidores = np.array([
    [2.0,   4.0,    50.0],
    [4.0,   8.0,   100.0],
    [8.0,  16.0,   250.0],
    [16.0, 32.0,   500.0],
    [32.0, 64.0,  1000.0],
    [64.0, 128.0, 2000.0]
])

nomes_servidores = [
    "Micro Instância Web",
    "Standard App Server",
    "Medium Backend & Cache",
    "Database Enterprise",
    "High Performance Computing",
    "Big Data & AI Training"
]

print(f"Catálogo: {catalogo_servidores.shape[0]} instâncias, "
      f"{catalogo_servidores.shape[1]} atributos técnicos.")

Catálogo: 6 instâncias, 3 atributos técnicos.


In [10]:
# ============================================================
# 3.1 DESAFIO 3 — FUNÇÃO DE RECOMENDAÇÃO
# ============================================================
def recomendar_servidores(catalogo, nomes, demanda, k=2):
    """
    Retorna os k servidores mais próximos da demanda (distância euclidiana).

    Retorna
    -------
    lista de tuplas (posicao, indice, nome, specs, distancia)
    """
    # Distância euclidiana vetorizada
    distancias = np.sqrt(np.sum((catalogo - demanda) ** 2, axis=1))

    # Ranking crescente
    ranking = np.argsort(distancias)[:k]

    return [
        (pos + 1, int(idx), nomes[idx], catalogo[idx], float(distancias[idx]))
        for pos, idx in enumerate(ranking)
    ]

In [11]:
# ============================================================
# 3.2 DESAFIO 3 — EXECUÇÃO E RANKING
# ============================================================
demanda = np.array([12.0, 28.0, 450.0])   # vCPUs, RAM (GB), SSD (GB)
K = 2

print("=" * 60)
print(f"RECOMENDAÇÃO DE SERVIDORES — Demanda: {demanda.tolist()}")
print("=" * 60)

resultados = recomendar_servidores(
    catalogo_servidores, nomes_servidores, demanda, k=K
)

for pos, idx, nome, specs, dist in resultados:
    print(f"\n{pos}º lugar: {nome}  (índice {idx})")
    print(f"   vCPUs : {specs[0]:.0f}")
    print(f"   RAM   : {specs[1]:.0f} GB")
    print(f"   SSD   : {specs[2]:.0f} GB")
    print(f"   Distância euclidiana à demanda: {dist:.2f}")

RECOMENDAÇÃO DE SERVIDORES — Demanda: [12.0, 28.0, 450.0]

1º lugar: Database Enterprise  (índice 3)
   vCPUs : 16
   RAM   : 32 GB
   SSD   : 500 GB
   Distância euclidiana à demanda: 50.32

2º lugar: Medium Backend & Cache  (índice 2)
   vCPUs : 8
   RAM   : 16 GB
   SSD   : 250 GB
   Distância euclidiana à demanda: 200.40
